In [3]:
import pandas as pd

df = pd.read_csv("data/metadata.csv")
print(df.columns)
df.head()


Index(['Work no.', 'Title', 'Author', 'Genre', 'PDF', 'Unicode'], dtype='object')


,Work no.,Title,Author,Genre,PDF,Unicode
0,1,திருக்குறள்,திருவள்ளுவர்,நீதிநெறி – பதினெண்கீழ்க்கணக்கு,pm0001.pdf pm0001_01.pdf,pmuni0001.html
1,2,ஆத்திசூடி,ஒளவையார்,நீதிநெறி நூல்கள்,pm0002.pdf,pmuni0002.html
2,2,கொன்றை வேந்தன்,ஒளவையார்,நீதிநெறி நூல்கள்,pm0002.pdf,pmuni0002.html
3,2,நல்வழி,ஒளவையார்,நீதிநெறி நூல்கள்,pm0002.pdf,pmuni0002.html
4,2,மூதுரை,ஒளவையார்,நீதிநெறி நூல்கள்,pm0002.pdf,pmuni0002.html


In [4]:
# defining Target poets

TARGET_POETS = ["ஒளவையார்", "பாரதிதாசன்"]


In [7]:
#filtering rows for only the chosen poets
df_filtered = df[df["Author"].isin(TARGET_POETS)]

print("Total poems selected:", len(df_filtered))
df_filtered["Author"].value_counts()


Total poems selected: 16


பாரதிதாசன்    12
ஒளவையார்       4
Name: Author, dtype: int64

In [34]:
def split_filenames(value):
    if pd.isna(value):
        return []
    return re.split(r"[,\s\u00A0]+", str(value).strip())




In [8]:
#pdf extraction 
import fitz  # PyMuPDF

pdf_path = "/data/pdf_poems"
def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text


In [9]:
#html extraction
from bs4 import BeautifulSoup

html_path= "/data/unicode_text"
def extract_html_text(html_path):
    with open(html_path, encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    return soup.get_text(separator="\n")


In [10]:
#cleaning tamil texts
import re

def clean_tamil_text(text):
    # Keep only Tamil Unicode + space + newline
    text = re.sub(r"[^\u0B80-\u0BFF\n ]", "", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


In [13]:
#Aggregation of poems per poet

from collections import defaultdict
import os
import re


PDF_COL = "PDF"      
HTML_COL = "Unicode"    
AUTHOR_COL = "Author"

def split_filenames(value):
    if pd.isna(value):
        return []
    return re.split(r"[,\s\u00A0]+", str(value).strip())

poet_corpus = defaultdict(list)

for _, row in df_filtered.iterrows():

    texts = []

    for pdf in split_filenames(row[PDF_COL]):
        path = os.path.join("data/pdf_poems", pdf)
        if os.path.exists(path):
            texts.append(extract_pdf_text(path))
        else:
            print(" Missing PDF:", path)

    for html in split_filenames(row[HTML_COL]):
        path = os.path.join("data/unicode_text", html)
        if os.path.exists(path):
            texts.append(extract_html_text(path))
        else:
            print(" Missing HTML:", path)

    combined = "\n".join(texts)
    clean = clean_tamil_text(combined)

    if len(clean) > 0:
        poet_corpus[row[AUTHOR_COL]].append(clean)


In [39]:
#creating .TXT file for fine tuning

os.makedirs("data/clean", exist_ok=True)

for poet, poems in poet_corpus.items():
    filename = poet + ".txt"   
    with open(os.path.join("data/clean", filename), "w", encoding="utf-8") as f:
        for poem in poems:
            f.write(poem + "\n\n")
